# **FINITE ELEMENTS in TIME in IRKSOME**

---

## **Install**

### Firedrake

In [7]:
try:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh"
    !bash "/tmp/firedrake-install.sh"
    from firedrake import *  # noqa: F401
except:
    from firedrake import *  # noqa: F401

--2026-06-04 12:16:40--  https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4767 (4.7K) [application/x-sh]
Saving to: ‘/tmp/firedrake-install.sh’

/tmp/firedrake-inst 100%[===================>]   4.66K  --.-KB/s    in 0s      

2026-06-04 12:16:40 (49.0 MB/s) - ‘/tmp/firedrake-install.sh’ saved [4767/4767]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FIREDRAKE_INSTALLED=/usr/local/share/fem-on-colab/firedrake.installed
+ [[ ! -f /usr/local/share/fem-on-colab/firedrake.installed ]]
+ set +x
























#############################################################

### Irksome

In [8]:
try:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git
    from irksome import *  # noqa: F401
except:
    from irksome import *  # noqa: F401

  Cloning https://github.com/firedrakeproject/Irksome.git to /tmp/pip-req-build-fxqsw3pl
  Running command git clone --filter=blob:none --quiet https://github.com/firedrakeproject/Irksome.git /tmp/pip-req-build-fxqsw3pl
  Resolved https://github.com/firedrakeproject/Irksome.git to commit b93c783549a96297e70fd18fa94d0a937997f08e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Other

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib import animation
from IPython.display import HTML, display

# Seaborn palette
seabornblue   = "#4C72B0"
seaborngreen  = "#55A868"
seabornred    = "#C44E52"
seabornpurple = "#8172B2"
seabornorange = "#F4953B"

---

## **Functions**

### $N$-body problem

In [15]:
# ============================================================================
# State space: N particles in 2D, positions q and momenta p. Irksome wants
# Firedrake `Function`s, so we use a trivial mesh and the "Real" element
# (globally constant DoFs) — the Function holds a 2N-vector with no spatial
# dependence whatsoever.
# ============================================================================
N = 5  # number of particles

eps2 = Constant(0.05**2)  # softening, squared

mesh = UnitIntervalMesh(1)
# Firedrake doesn't currently support vector-valued "Real" spaces (no Argument
# allowed). DG(0) on a single-cell mesh is equivalent: one DoF per component,
# and integration over the unit-measure cell just returns the value.
Vq = VectorFunctionSpace(mesh, "DG", 0, dim=2*N)
Vp = VectorFunctionSpace(mesh, "DG", 0, dim=2*N)
Z = Vq * Vp

state = Function(Z)
q, p = split(state)
vq, vp = TestFunctions(Z)

def vec(field, i):
    """Extract the i-th particle's 2D sub-vector from a flat 2N-vector."""
    return as_vector([field[2*i], field[2*i+1]])

# ============================================================================
# Hamiltonian as a UFL expression — used for diagnostics.
# ============================================================================
def kinetic(p_field):
    return sum([
        inner(vec(p_field, i), vec(p_field, i)) / 2
        for i in range(N)
    ])

def potential(q_field):
    return - sum([
        1 / sqrt(inner(vec(q_field, i) - vec(q_field, j),
                       vec(q_field, i) - vec(q_field, j)) + eps2)
        for i in range(N) for j in range(i+1, N)
    ])

H_form = (kinetic(p) + potential(q)) * dx(degree=0)

# ============================================================================
# Variational form encoding Hamilton's equations.
#   dq/dt - p     = 0,  tested against vq
#   dp/dt + dU/dq = 0,  tested against vp
# ============================================================================
F = (
    inner(Dt(q) - p, vq)
  + inner(Dt(p), vp)
) * dx(degree=0)

for i in range(N):
    for j in range(N):
        if j == i: continue
        diff = vec(q, i) - vec(q, j)
        F += inner(diff, vec(vp, i)) / sqrt(inner(diff, diff) + eps2)**3 * dx(degree=0)

# ============================================================================
# Initial conditions: positions and momenta drawn from a Gaussian, with
# momenta shifted so the total system momentum vanishes (centre of mass
# stays put).
# ============================================================================
sigma_q = 1.0    # std dev of initial positions
sigma_p = 0.0    # std dev of initial momenta

def set_initial_conditions(seed=0):
    rng = np.random.default_rng(seed)
    q0 = rng.standard_normal(2*N) * sigma_q
    p0 = rng.standard_normal(2*N) * sigma_p
    p0 = p0.reshape(N, 2)
    p0 -= p0.mean(axis=0)
    p0 = p0.reshape(-1)
    state.sub(0).dat.data[:] = q0
    state.sub(1).dat.data[:] = p0

# ============================================================================
# Time-stepping driver — takes a scheme, returns a dict of time series for
# plotting.
# ============================================================================
solver_params = {
    "snes_type": "newtonls",
    "snes_rtol": 1e-12,
    "snes_atol": 1e-12,
    "ksp_type": "preonly",
    "pc_type": "lu",
}

def run(scheme, T=2.0, dt=0.01, seed=0):
    set_initial_conditions(seed)
    Nt      = round(T / dt)
    t       = Constant(0.0)
    stepper = TimeStepper(F, scheme, t, dt, state,
                          solver_parameters=solver_params)

    times     = np.zeros(Nt + 1)
    positions = np.zeros((Nt + 1, 2*N))
    energies  = np.zeros(Nt + 1)
    positions[0] = state.sub(0).dat.data_ro.copy()
    energies[0]  = float(assemble(H_form))

    for k in range(Nt):
        stepper.advance()
        t.assign(float(t) + float(dt))
        times[k+1]     = float(t)
        positions[k+1] = state.sub(0).dat.data_ro.copy()
        energies[k+1]  = float(assemble(H_form))

    return {"times": times, "positions": positions, "energies": energies}

# ============================================================================
# Plotting helpers.
# ============================================================================
_palette = {"CPG(1)": seaborngreen, "GL(1)": seabornred}

def plot_energy_drift(results):
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, res in results.items():
        rel = res["energies"] - res["energies"][0]
        ax.plot(res["times"], rel, label=name,
                color=_palette.get(name), lw=1.5)
    ax.set_xlabel("$t$")
    ax.set_ylabel(r"$H(t) - H(0)$")
    ax.set_title("Energy error")
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.legend()
    fig.tight_layout()
    plt.show()

def plot_orbits_grid(results, frames=200, interval=40, tail_time=None):
    """Animate orbit trails side-by-side. Returns an inline HTML video.

    Parameters
    ----------
    tail_time : float or None
        If given, only the most recent `tail_time` time units of trail are
        shown at each frame (older segments are dropped). The colour gradient
        is normalised within the visible window: newest = blue, oldest = yellow.
        If None, the full trail is shown and the gradient spans the whole run.
    """
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]

    # Shared axis limits so the panels can be compared directly.
    all_pos = np.concatenate([r["positions"] for r in results.values()], axis=0)
    all_x = all_pos[:, 0::2]
    all_y = all_pos[:, 1::2]
    pad = 0.1 * max(np.ptp(all_x), np.ptp(all_y))
    xlim = (all_x.min() - pad, all_x.max() + pad)
    ylim = (all_y.min() - pad, all_y.max() + pad)

    # Frame schedule: subsample the time grid down to `frames` frames.
    times_global = next(iter(results.values()))["times"]
    n_steps  = len(times_global)
    n_frames = min(frames, n_steps)
    frame_idx = np.linspace(0, n_steps - 1, n_frames).astype(int)

    # Reversed viridis: 0 -> yellow (back of tail), 1 -> blue (front).
    cmap = plt.get_cmap("viridis_r")
    norm = plt.Normalize(0.0, 1.0)

    line_collections = []  # outer: per subplot; inner: per particle
    for ax, (name, res) in zip(axes, results.items()):
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect("equal")
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])
        subplot_lcs = []
        for _ in range(N):
            lc = LineCollection([], cmap=cmap, norm=norm, lw=1.5, alpha=0.9)
            ax.add_collection(lc)
            subplot_lcs.append(lc)
        line_collections.append(subplot_lcs)

    def update(k):
        idx = frame_idx[k]
        t_now    = times_global[idx]
        t_oldest = times_global[0] if tail_time is None else t_now - tail_time
        artists = []
        for (_name, res), subplot_lcs in zip(results.items(), line_collections):
            positions = res["positions"]
            times     = res["times"]
            start = int(np.searchsorted(times, t_oldest))
            window = t_now - t_oldest
            for i, lc in enumerate(subplot_lcs):
                if idx > start and window > 0:
                    x = positions[start:idx + 1, 2*i]
                    y = positions[start:idx + 1, 2*i+1]
                    pts  = np.array([x, y]).T.reshape(-1, 1, 2)
                    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
                    lc.set_segments(segs)
                    # Colour each segment by its position within the tail window.
                    seg_t = times[start:idx]
                    lc.set_array((seg_t - t_oldest) / window)
                else:
                    lc.set_segments([])
                artists.append(lc)
        return artists

    fig.tight_layout()
    ani = animation.FuncAnimation(
        fig, update, frames=n_frames, interval=interval, blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# ============================================================================
# Run it all.
# ============================================================================
def kepler_results(**params):
    return {
        "CPG(1)" : run(GalerkinCollocationScheme(1, quadrature_degree="auto"), **params),
        "GL(1)"  : run(GaussLegendre(1), **params),
    }

### Nonlinear Schrödinger

In [11]:
# ============================================================================
# Plotting helpers for the NLS demo.
# ============================================================================

def plot_drifts(results):
    """Energy drift (left) + mass drift (right), both schemes overlaid."""
    fig, (axE, axM) = plt.subplots(1, 2, figsize=(13, 4))
    palette = {"CPG(1)": seaborngreen, "GL(1)": seabornred}
    for name, res in results.items():
        c = palette.get(name)
        axE.plot(res["times"], res["energies"] - res["energies"][0],
                 label=name, color=c, lw=1.5)
        axM.plot(res["times"], res["masses"] - res["masses"][0],
                 label=name, color=c, lw=1.5)
    for ax, label, title in [(axE, "$E(t) - E(0)$", "Energy drift"),
                              (axM, "$M(t) - M(0)$", "Mass drift")]:
        ax.set_xlabel("$t$")
        ax.set_ylabel(label)
        ax.axhline(0, color="grey", lw=0.5, ls="--")
        ax.set_title(title)
        ax.legend()
    fig.tight_layout()
    plt.show()


def plot_psi_animation(results, x_coords, frames=200, interval=40):
    """Animate |ψ(x, t)|² side-by-side, one panel per scheme."""
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]

    # Shared y-axis based on max |ψ|² across both schemes.
    all_snaps = np.concatenate([r["snapshots"] for r in results.values()])
    ymax = 1.1 * float(all_snaps.max())

    # Frame schedule.
    times_global = next(iter(results.values()))["times"]
    n_steps  = len(times_global)
    n_frames = min(frames, n_steps)
    frame_idx = np.linspace(0, n_steps - 1, n_frames).astype(int)

    lines, titles = [], []
    for ax, name in zip(axes, results.keys()):
        ax.set_xlim(x_coords.min(), x_coords.max())
        ax.set_ylim(0, ymax)
        ax.set_xlabel("$x$")
        ax.set_ylabel(r"$|\psi(x, t)|^2$")
        line, = ax.plot([], [], color=seabornblue, lw=2)
        title = ax.set_title(f"{name}")
        lines.append(line)
        titles.append(title)

    def update(k):
        idx = frame_idx[k]
        artists = []
        for line, title, (name, res) in zip(lines, titles, results.items()):
            line.set_data(x_coords, res["snapshots"][idx])
            title.set_text(f"{name}    $t = {res['times'][idx]:.2f}$")
            artists.extend([line, title])
        return artists

    fig.tight_layout()
    ani = animation.FuncAnimation(
        fig, update, frames=n_frames, interval=interval, blit=False,
    )
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

---

## **TL;DR**

I'm going to be discussing the new support for **Galerkin in time** / **finite elements in time** / **variational in time integrators** in Irksome.

This includes both continuous Petrov–Galerkin (CPG) and discontinuous Galerkin (DG) scheme (I'll mostly discuss the former though, since they're my favourite).
Just like the rest of Irksome, these both interact with the standard Firedrake / PETSc solver interface (incl. multigrid, fieldsplit, etc.).

I'll open with a bit of discussion of ODEs for didactic purposes — *(just to make sure everyone's on the same page w.r.t. CPG)* — before moving onto a PDE demo to show how the interface work

---

## **Finite elements in time for ODEs**

To introduce the concept of variational-in-time methods, we're going to keep it simple to start, looking just at ODEs.

### General framework

Take an ODE for $x : [0, T] \to \mathbb{R}$ (or $\mathbf{x} : [0, T] \to \mathbb{R}^n$):
> $$
> \dot{x} = f(x).
> $$

Suppose we have some initial data $x(t_n) = x_n$ and want to advance to $t_{n+1} = t_n + \Delta t$.

A very general idea for an $s$-stage time discretisation is as follows: *seek a polynomial $x_h$ of degree $s$ on $[t_n, t_{n+1}]$ such that*
1. *$x_h(t_n) = x_n$ (interpolate the initial data)*
2. *$s$ further constraints, aiming to capture the ODE $\dot{x} = f(x)$*

So, how do we pick those $s$ further constraints?

### Option 1: Simple collocation $\to$ RK

Require that $\dot{x}_h = f(x_h)$ holds *exactly* at $s$ chosen *collocation points* $\xi_1, \ldots, \xi_s \in [t_n, t_{n+1}]$:
> $$
> \dot{x}_h(\xi_k) = f(x_h(\xi_k)), \qquad k = 1, \ldots, s.
> $$

This recovers **collocation Runge–Kutta (RK) methods**:
- $s = 1$, midpoint node $\implies$ implicit midpoint
- $s$ Gauss–Legendre nodes $\implies$ Gauss–Legendre RK
- $s$ right-Radau nodes $\implies$ Radau IIA

### Option 2: Test against polynomial test functions $\to$ CPG

Let's instead put on our FEM hats for a second...

Rather than enforcing the ODE *pointwise* at $s$ nodes, we could enforce it *weakly* against polynomial test functions $y_h \in \mathcal{P}_{s-1}(t_n, t_{n+1})$:
> $$
> \int_{t_n}^{t_{n+1}} \dot{x}_h \,y_h\, \mathrm{d}t = \int_{t_n}^{t_{n+1}} f(x_h) \,y_h\, \mathrm{d}t, \qquad \forall y_h \in \mathcal{P}_{s-1}.
> $$

The mismatch in polynomial degrees (solution in $\mathcal{P}_s$, tests in $\mathcal{P}_{s-1}$) is by design. One of the solution's $s+1$ DoFs has already been spent on the initial condition, leaving $s$ unknowns to match the $s$-dimensional test space.

This defines the **continuous Petrov–Galerkin** (CPG) method:
- *"Continuous"*: the solution function is continuous across timesteps (initial data is strongly enforced).
- *"Petrov–Galerkin"*: the trial and test spaces differ in polynomial degree.

*(There's also a bit of a sibling FET scheme, discontinuous Galerkin (DG). We won't discuss this here today, but needless to say it's also now available in Irksome.)*

### So... Why would this ever be useful?

- **CPG generalises collocation RK.**
Approximating the integrals $\int_{t_n}^{t_{n+1}}$ above using an $s$-node quadrature rule, we recover *exactly* the corresponding collocation RK method.
So we don't lose anything by switching to CPG.

- **CPG generalises schemes beyond collocation RK.**
There are schemes that don't fit naturally in the language of RK methods, but do in the language of CPG with quadrature.
For example, one gets Crank-Nicolson *(distinct from implicit midpoint for nonlinear systems!)* with (i) $s=1$, and (ii) $\int_{t_n}^{t_{n+1}}$ approximated by the trapezium rule.

- **Many ideas from RK methods fit naturally in this language.**
E.g. (i) split methods are just the approximation of different integrals $\int_{t_n}^{t_{n+1}}$ on different terms in your residual by different quadrature rules, while (ii) certain stage-coupled RK preconditioners are equivalently just choices of bases in CPG land.

- **Structure preservation in CPG.**
The exact integrals in CPG methods give them some *great* conservative and dissipative properties.
For example, CPG is *energy-conserving* for Hamiltonian systems.
This all boils down to being able to reproduce the fundamental theorem of calculus, $\mathcal{H}(t_{n+1}) - \mathcal{H}(t_n) = \int_{t_n}^{t_{n+1}} \dot{\mathcal{H}}$.
The proof's relatively simple...

### CPG is energy-conserving for Hamiltonian systems

Take a Hamiltonian system in $(\mathbf{q}, \mathbf{p}) \in \mathbb{R}^{2n}$:
> $$
> \dot{\mathbf{q}} = \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}, \mathbf{p}), \qquad \dot{\mathbf{p}} = - \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}, \mathbf{p}).
> $$

Over $[t_n, t_{n+1}]$, CPG seeks $(\mathbf{q}_h, \mathbf{p}_h) \in \mathcal{P}_s^{2n}$ satisfying initial data $\mathbf{q}_h(t_n) = \mathbf{q}_n$ and $\mathbf{p}_h(t_n) = \mathbf{p}_n$, and such that
> $$
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{q}}_h \cdot \mathbf{r}_h = \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \mathbf{r}_h, \qquad \int_{t_n}^{t_{n+1}} \dot{\mathbf{p}}_h \cdot \mathbf{s}_h = - \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \mathbf{s}_h,
> $$

for all test functions $(\mathbf{r}_h, \mathbf{s}_h) \in \mathcal{P}_{s-1}^{2n}$.

The key observation is that $\dot{\mathbf{q}}_h$ and $\dot{\mathbf{p}}_h$ are themselves degree-$(s-1)$ polynomials, so we are free to test the first equation with $\mathbf{r}_h = \dot{\mathbf{p}}_h$ and the second with $\mathbf{s}_h = \dot{\mathbf{q}}_h$. This gives
> $$
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{q}}_h \cdot \dot{\mathbf{p}}_h = \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \dot{\mathbf{p}}_h, \qquad \int_{t_n}^{t_{n+1}} \dot{\mathbf{p}}_h \cdot \dot{\mathbf{q}}_h = - \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \dot{\mathbf{q}}_h,
> $$

Summing and applying the chain rule,
> $$
> \mathcal{H}(t_{n+1}) - \mathcal{H}(t_n)
> = \int_{t_n}^{t_{n+1}} \dot{\mathcal{H}}
> = \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{q}} \cdot \dot{\mathbf{q}}_h + \frac{\partial \mathcal{H}}{\partial \mathbf{p}} \cdot \dot{\mathbf{p}}_h
> = 0,
> $$

So, provided the time integrals are exact, $\mathcal{H}$ is preserved across timesteps regardless of time step size.
Similar results exist for other systems (e.g. gradient descent), while *"discrete gradients"* show us that any conservation law/dissipation inequality can be preserved discretely with a well-chosen mixed CPG discretisation.

### $N$-body problem

As a little motivating demo, let's evolve $N = 5$ particles in 2D moving under their own mutual gravity.
This is a Hamiltonian system, with
> $$
> \mathcal{H} = \sum_{i=1}^{N} \frac{\|\mathbf{p}_i\|^2}{2} - \sum_{i<j} \frac{1}{\sqrt{\|\mathbf{q}_i - \mathbf{q}_j\|^2}}.
> $$

We'll compare results from from two schemes:
- **CPG(1):** 1-stage CPG (preserves the Hamiltonian)
- **GL(1):** 1-stage Gauss–Legendre, i.e. implicit midpoint (symplectic, but only preserves *quadratic* invariants exactly)

In [ ]:
results = kepler_results(dt=1e-3)
plot_energy_drift(results)
plot_orbits_grid(results, tail_time=0.4)

In [ ]:
results = kepler_results(dt=5e-3)
plot_energy_drift(results)
plot_orbits_grid(results, tail_time=0.4)

---

## **Finite elements in time for PDEs**

With the ODE picture set up, we can finally turn our attention to PDEs...

### From ODEs to PDEs

When constructing finite element timestepping schemes, we *always* begin by ignoring the fact there's time derivative, and discretising the spatial component in the standard variational way.
This is called a **semidiscretisation** (in space only).
This is typically something of the form:
> *Find $u^h \in U^h$ (a finite element space) such that*
> $$M(\dot{u}^h, v^h) = F(u^h; v^h)$$
> for all $v^h \in U^h$.

This then gives us a big coupled ODE in time over the spatial degrees of freedom.
Just as above, we can evolve this ODE how we choose, either via a RK method, or finite elements in time, e.g. CPG.

After all that, CPG essentially just boils down to what you would have got had you done a **variational discretisation over the whole space-time slab** $\Omega \times [t_n, t_{n+1}]$.
*Time is just another dimension!*
Given we're already doing finite element in space, the finite-element-in-time machinery slots in right next to the spatial discretisation without much extra thinking.

Many of the structure-preservation properties I mention above then carry over verbatim.
In particular, the energy-conservation proof for Hamiltonian systems from above generalises directly.

### How we do this in Irksome

<!-- A brief technical aside, because it determines what solvers work and which don't. -->

To assemble the CPG variational problem on a slab, you (the user) picks:
- A basis $(\phi_0, \phi_1, \ldots, \phi_s)$ for $\mathcal{P}_s(t_n, t_{n+1})$;
- A quadrature rule / degree, to approximate $\int_{t_n}^{t_{n+1}}$.

Behind the scenes, the discrete state is stored as a tuple $(u_0, u_1, \ldots, u_s)$ of spatial `Function`s, one for each time-basis function $(\phi_0, \phi_1, \ldots, \phi_s)$.
Mapping these to values at the temporal quadrature points $(\xi_1, \xi_2, \ldots, \xi_N)$ is then just pre-multiplication by the matrix
> $$
> \begin{pmatrix}
> \phi_0(\xi_1) & \phi_1(\xi_1) & \cdots & \phi_s(\xi_1) \\
> \phi_0(\xi_2) & \phi_1(\xi_2) & \cdots & \phi_s(\xi_2) \\
> \vdots        & \vdots        & \ddots & \vdots        \\
> \phi_0(\xi_N) & \phi_1(\xi_N) & \cdots & \phi_s(\xi_N) \\
> \end{pmatrix};
> $$

mapping to time-derivative values is just pre-multiplication by
> $$
> \begin{pmatrix}
> \phi'_0(\xi_1) & \phi'_1(\xi_1) & \cdots & \phi'_s(\xi_1) \\
> \phi'_0(\xi_2) & \phi'_1(\xi_2) & \cdots & \phi'_s(\xi_2) \\
> \vdots         & \vdots         & \ddots & \vdots         \\
> \phi'_0(\xi_N) & \phi'_1(\xi_N) & \cdots & \phi'_s(\xi_N) \\
> \end{pmatrix}.
> $$

The Irksome interface is then super similar to that of standard RK methods.
Whereas normally a timestepping scheme would be specified by e.g. `scheme = GaussLegendre(degree)`, one can access CPG schemes via `scheme = GalerkinCollocationScheme(degree)`.
Bases in time and quadrature rules are passed as parameters to `GalerkinCollocationScheme`, e.g. `scheme = GalerkinCollocationScheme(degree, quadrature_degree=..., quadrature_scheme=...)`.

### Nonlinear Schrödinger

To show you this in practice, we'll take a Hamiltonian PDE example:
the nonlinear Schrödinger equation,
> $$
> i \, \partial_t \psi = -\frac{1}{2} \partial_{xx} \psi + \beta \, |\psi|^2 \psi,
> $$

on a periodic 1D domain.
This has two conserved quantities:

- **Mass**: $\displaystyle \mathcal{M}(\psi) = \int_\Omega |\psi|^2$, *quadratic* (in $\psi$)
- **Energy**: $\displaystyle \mathcal{E}(\psi) = \int_\Omega \tfrac{1}{2} |\partial_x \psi|^2 + \tfrac{1}{2} \beta |\psi|^4$, *non-quadratic*

We'll discretise in space with the standard real–imaginary split $\psi = u + i v$ (so we don't need to stop and re-install a complex Firedrake build), and run both CPG and a Gauss–Legendre RK method on the same initial data.
We expect to see CPG preserve $\mathcal{E}$ at the cost of some drift in $\mathcal{M}$;
Gauss should do the opposite *(neither preserves both for this example)*.

In [ ]:
# ============================================================================
# Basic mesh / function space setup
# ============================================================================
L  = 20.0  # domain half-width
nx = 200  # no. of cells
beta = Constant(1.0)

mesh = PeriodicIntervalMesh(nx, 2*L)
V    = FunctionSpace(mesh, "CG", 1)
Z    = V * V

state = Function(Z)
u, v  = split(state)
w, z  = TestFunctions(Z)

x, = SpatialCoordinate(mesh)
x -= L  # recentre to [-L, L]

T  = 4.0
dt = 0.02
Nt = round(T / dt)

In [ ]:
# ============================================================================
# Initial condition: moving wave packet ψ(x, 0) = exp(-(x-x₀)²/σ²) exp(ikx)
# ============================================================================
x0    = Constant(-5.0)
sigma = Constant(2.0)
k     = Constant(2.0)

envelope = exp(- (x - x0)**2 / sigma**2)

def set_initial_conditions():
    state.sub(0).interpolate(envelope * cos(k * x))
    state.sub(1).interpolate(envelope * sin(k * x))

In [ ]:
# ============================================================================
# NLS residual in (u, v) form: ∂ₜu = -½ ∂ₓₓv + β|ψ|²v, ∂ₜv = ½ ∂ₓₓu - β|ψ|²u
# ============================================================================
psi2 = u**2 + v**2
F = (
    inner(Dt(u), w) - 0.5 * inner(grad(v), grad(w)) - beta * psi2 * v * w
  + inner(Dt(v), z) + 0.5 * inner(grad(u), grad(z)) + beta * psi2 * u * z
) * dx

In [ ]:
# ============================================================================
# Output, incl. invariants
# ============================================================================
M_form = psi2 * dx
H_form = (
    0.5 * (inner(grad(u), grad(u)) + inner(grad(v), grad(v)))
  + 0.5 * beta * psi2**2
) * dx

# Sorted spatial DoF coordinates for plotting
x_coords = V.tabulate_dof_coordinates().flatten()
sort_idx = np.argsort(x_coords)
x_coords = x_coords[sort_idx]

def record():
    u_data = state.sub(0).dat.data_ro
    v_data = state.sub(1).dat.data_ro
    return (u_data**2 + v_data**2)[sort_idx]

results = {}

In [ ]:
# ============================================================================
# The magic one-line scheme switch!
# ============================================================================
# HERE ARE THE DIFFERENT SCHEMES...
schemes = {
    "CPG(1)": GalerkinCollocationScheme(1, quadrature_degree="auto"),
    "GL(1)":  GaussLegendre(1),
}

for name, scheme in schemes.items():
    set_initial_conditions()
    t = Constant(0.0)

    # ...THAT WE JUST PLUG IN HERE
    stepper = TimeStepper(F, scheme, t, Constant(dt), state)

    times     = np.zeros(Nt + 1)
    energies  = np.zeros(Nt + 1)
    masses    = np.zeros(Nt + 1)
    snapshots = np.zeros((Nt + 1, V.dim()))

    energies[0]  = float(assemble(H_form))
    masses[0]    = float(assemble(M_form))
    snapshots[0] = record()

    for j in range(Nt):
        stepper.advance()
        t.assign(float(t) + dt)
        times[j+1]     = float(t)
        energies[j+1]  = float(assemble(H_form))
        masses[j+1]    = float(assemble(M_form))
        snapshots[j+1] = record()

    results[name] = {
        "times": times,   "energies": energies,
        "masses": masses, "snapshots": snapshots,
    }

In [ ]:
plot_drifts(results)

In [ ]:
plot_psi_animation(results, x_coords)

---

## **Some more features!**

Some honourable mentions of features I've not demoed here, but are available in the Irksome package...

### Differing quadrature on different terms

One detail here that's important to the story...

The `quadrature_degree="auto"` keyword on `GalerkinCollocationScheme` above made the time quadrature **exact** *(or as good as it can be if things hadn't been polynomial)*.
We did this because we needed exact integrals for energy conservation.

More generally however, this might not be what you want.
In particular, you might hypothetically only want / need a high-degree quadrature rule on a couple terms, and be willing to settle for some other $s$-stage quadrature on others.

*You can do this kind of splitting in Irksome!*

For example, if I only wanted $s$-stage Gauss–Legendre quadature on the term `inner(grad(u), grad(z)) * dx`, I could mark this with a `TimeQuadratureLabel` object:
```
Llow = TimeQuadratureLabel(2*s-1, scheme="gauss")
F = ... + Llow(inner(grad(u), grad(z)) * dx) + ...
```

This is **very** useful in terms of computational efficiency, but in this context it's purely didactic.
Everything here is polynomial, and we want to integrate exactly, so `quadrature_degree="auto"` is what we want.

### Discontinuous Galerkin

As I mention, CPG isn't the only Galerkin-in-time scheme Irksome supports.
Discontinuous Galerkin (DG) has many merits, and is potentially better studied than CPG.
Alongside `GalerkinCollocationScheme`, you've got `DiscontinuousGalerkinCollocationScheme`.
The `TimeStepper` interface is identical.

### Solvers (everything we're used to from PETSc)

At the end of the day, behind the scenes, the stage-coupled CPG / DG problem lands as a plain Firedrake nonlinear residual on a `MixedFunctionSpace`.
Just like the rest of Irksome, it composes therefore with the full PETSc solver stack: Newton, Krylov, fieldsplit, multigrid, etc.
The same `solver_parameters = {...}` dictionary pattern you'd use for any other Firedrake problem works here.

### Discontinuous auxiliary variables for CPG

It's a little fiddly to fit differential–algebraic equations (e.g. Stokes) into CPG.
You ***really*** ought not to model the algebraic variables (e.g. the pressure $p$) as continuous-in-time too;
being a bit loosey goosey and treating them equivalently ends up not really affecting your solution for RK methods, but it *does* for CPG.

The solution is to make these variables discontinuous-in-time, putting them in $\mathcal{P}_{s-1}$ (where the continuous-in-time primal variables are in $\mathcal{P}_s$).

This can be done with a single `aux_indices = […]` parameter on `TimeStepper` *(e.g. for Stokes you would want `TimeStepper(..., aux_indices = [1])`)*.

Anyone who's seen me talk at all in the past 2 years will know that I *love* this.
Using these discontinuous-in-time auxiliary variables with CPG lets you preserve *pretty much* any conservation law / dissipation inequality you want.
But that's a story for another time...